In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# --- AŞAMA 1: GÖRÜNTÜYÜ OKU VE HAZIRLA ---
# Görüntüyü gri tonlamalı oku (Kenar tespiti için renk gerekmez)
img = cv2.imread('ornek_goruntu.png', cv2.IMREAD_GRAYSCALE)

# --- AŞAMA 2: TEMEL HESAPLAMALAR ---

# 2.1. Gauss Blur Uygulanmış Görüntü (Temizlik)
# (5, 5) kernel boyutu, gürültüyü yumuşatmak için idealdir.
blurred = cv2.GaussianBlur(img, (5, 5), 0)

# --- AŞAMA 3: SOBEL OPERASYONLARI (X, Y, XY) ---

# 3.1. Orijinal Gürültülü Görüntüye Sobel (Ham)
sobelx_raw = cv2.Sobel(img, cv2.CV_64F, 1, 0, ksize=3) # Sadece dikey kenarlar
sobely_raw = cv2.Sobel(img, cv2.CV_64F, 0, 1, ksize=3) # Sadece yatay kenarlar
sobelx_raw_8bit = cv2.convertScaleAbs(sobelx_raw) # Mutlak değer ve 8-bit'e sabitleme
sobely_raw_8bit = cv2.convertScaleAbs(sobely_raw)
# Ham Sobel X + Y (Gürültülü Tam Kenar Haritası)
sobelxy_raw_8bit = cv2.addWeighted(sobelx_raw_8bit, 0.5, sobely_raw_8bit, 0.5, 0)

# 3.2. Gauss Uygulanmış Görüntüye Sobel 
sobelx_blur = cv2.Sobel(blurred, cv2.CV_64F, 1, 0, ksize=3) # Temiz dikey kenarlar
sobely_blur = cv2.Sobel(blurred, cv2.CV_64F, 0, 1, ksize=3) # Temiz yatay kenarlar
sobelx_blur_8bit = cv2.convertScaleAbs(sobelx_blur) # Mutlak değer ve 8-bit'e sabitleme
sobely_blur_8bit = cv2.convertScaleAbs(sobely_blur)
# Gauss + Sobel X + Y
sobelxy_blur_8bit = cv2.addWeighted(sobelx_blur_8bit, 0.5, sobely_blur_8bit, 0.5, 0)

# --- AŞAMA 4: 9'LU PANEL GÖRSELLEŞTİRME ---

plt.figure(figsize=(18, 15))

# --- BİRİNCİ SATIR: HAM GÖRÜNTÜ VE YÖNLÜ SOBEL ---
plt.subplot(3, 3, 1); plt.imshow(img, cmap='gray'); plt.title("1. Orijinal (Gürültülü)"); plt.axis('off')
plt.subplot(3, 3, 2); plt.imshow(sobelx_raw_8bit, cmap='gray'); plt.title("2. Sobel X (Dikey Kenarlar)"); plt.axis('off')
plt.subplot(3, 3, 3); plt.imshow(sobely_raw_8bit, cmap='gray'); plt.title("3. Sobel Y (Yatay Kenarlar)"); plt.axis('off')

# --- İKİNCİ SATIR: GAUSS TEMİZLİĞİ VE YÖNLÜ SOBEL ---
plt.subplot(3, 3, 4); plt.imshow(blurred, cmap='gray'); plt.title("4. Gauss Blur Uygulanmış (Temiz)"); plt.axis('off')
plt.subplot(3, 3, 5); plt.imshow(sobelx_blur_8bit, cmap='gray'); plt.title("5. Gauss + Sobel X"); plt.axis('off')
plt.subplot(3, 3, 6); plt.imshow(sobely_blur_8bit, cmap='gray'); plt.title("6. Gauss + Sobel Y"); plt.axis('off')

# --- ÜÇÜNCÜ SATIR: HAM VE TEMİZ SOBEL XY (BİRLEŞTİRİLMİŞ) ---
plt.subplot(3, 3, 7); plt.imshow(img, cmap='gray'); plt.title("7. Orijinal (Gürültülü) (Tekrar)"); plt.axis('off')
plt.subplot(3, 3, 8); plt.imshow(sobelxy_raw_8bit, cmap='gray'); plt.title("8. Ham Sobel XY (Gürültülü)"); plt.axis('off')
plt.subplot(3, 3, 9); plt.imshow(sobelxy_blur_8bit, cmap='gray'); plt.title("9. Gauss + Sobel XY "); plt.axis('off')

# Panelin yerleşimini sıkıştır
plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# --- AŞAMA 1: GÖRÜNTÜYÜ OKU ---
img = cv2.imread('ornek_goruntu.png', cv2.IMREAD_GRAYSCALE)

if img is None:
    # Test için gürültülü sentetik görüntü
    img = np.zeros((300, 300), dtype=np.uint8)
    cv2.rectangle(img, (80, 80), (220, 220), 180, -1)
    noise = np.random.randint(0, 35, (300, 300), dtype=np.uint8)
    img = cv2.add(img, noise)

# --- AŞAMA 2: ÖN HAZIRLIK ---
# Gauss Blur (5,5) - Scharr çok hassas olduğu için gürültüyü çok fazla büyütür.
blurred = cv2.GaussianBlur(img, (5, 5), 0)

# --- AŞAMA 3: SCHARR OPERASYONLARI ---

# 3.1. Ham Görüntü Üzerinde Scharr (Gürültüye Duyarlılık Testi)
# Scharr fonksiyonunda ksize parametresi yoktur, doğrudan çalışır.
scharrx_raw = cv2.Scharr(img, cv2.CV_64F, 1, 0)
scharry_raw = cv2.Scharr(img, cv2.CV_64F, 0, 1)
scharrx_raw_8bit = cv2.convertScaleAbs(scharrx_raw)
scharry_raw_8bit = cv2.convertScaleAbs(scharry_raw)
scharrxy_raw_8bit = cv2.addWeighted(scharrx_raw_8bit, 0.5, scharry_raw_8bit, 0.5, 0)

# 3.2. Gauss Uygulanmış Görüntü Üzerinde Scharr (İdeal Kullanım)
scharrx_blur = cv2.Scharr(blurred, cv2.CV_64F, 1, 0)
scharry_blur = cv2.Scharr(blurred, cv2.CV_64F, 0, 1)
scharrx_blur_8bit = cv2.convertScaleAbs(scharrx_blur)
scharry_blur_8bit = cv2.convertScaleAbs(scharry_blur)
scharrxy_blur_8bit = cv2.addWeighted(scharrx_blur_8bit, 0.5, scharry_blur_8bit, 0.5, 0)

# --- AŞAMA 4: 9'LU PANEL GÖRSELLEŞTİRME ---
plt.figure(figsize=(18, 15))

# ÜST SATIR: Ham Görüntü ve Yönlü Scharr
plt.subplot(3, 3, 1); plt.imshow(img, cmap='gray'); plt.title("1. Orijinal (Gürültülü)"); plt.axis('off')
plt.subplot(3, 3, 2); plt.imshow(scharrx_raw_8bit, cmap='gray'); plt.title("2. Scharr X (Ham)"); plt.axis('off')
plt.subplot(3, 3, 3); plt.imshow(scharry_raw_8bit, cmap='gray'); plt.title("3. Scharr Y (Ham)"); plt.axis('off')

# ORTA SATIR: Gauss Blur ve Yumuşatılmış Scharr
plt.subplot(3, 3, 4); plt.imshow(blurred, cmap='gray'); plt.title("4. Gauss Blur Uygulanmış"); plt.axis('off')
plt.subplot(3, 3, 5); plt.imshow(scharrx_blur_8bit, cmap='gray'); plt.title("5. Gauss + Scharr X"); plt.axis('off')
plt.subplot(3, 3, 6); plt.imshow(scharry_blur_8bit, cmap='gray'); plt.title("6. Gauss + Scharr Y"); plt.axis('off')

# ALT SATIR: XY Birleşimleri
plt.subplot(3, 3, 7); plt.imshow(img, cmap='gray'); plt.title("7. Orijinal (Referans)"); plt.axis('off')
plt.subplot(3, 3, 8); plt.imshow(scharrxy_raw_8bit, cmap='gray'); plt.title("8. Ham Scharr XY (Çok Gürültülü)"); plt.axis('off')
plt.subplot(3, 3, 9); plt.imshow(scharrxy_blur_8bit, cmap='gray'); plt.title("9. Gauss + Scharr XY"); plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# --- AŞAMA 1: GÖRÜNTÜYÜ OKU ---
img = cv2.imread('ornek_goruntu.png', cv2.IMREAD_GRAYSCALE)

if img is None:
    # Test için gürültülü sentetik görüntü
    img = np.zeros((300, 300), dtype=np.uint8)
    cv2.rectangle(img, (80, 80), (220, 220), 180, -1)
    noise = np.random.randint(0, 35, (300, 300), dtype=np.uint8)
    img = cv2.add(img, noise)

# --- AŞAMA 2: ÖN HAZIRLIK ---
# İkinci dereceden türev gürültüyü çok fazla büyüteceği için
# Gauss Blur (5,5) veya daha yüksek bir kernel ile uygulanması şarttır.
blurred = cv2.GaussianBlur(img, (5, 5), 0)

# --- AŞAMA 3: LAPLACIAN OPERASYONLARI ---
# Laplacian yönsüz olduğu için X ve Y olarak ayrılmaz.

# 3.1. Ham Görüntü Üzerinde Laplacian (Gürültüye Duyarlılık Testi)
laplacian_raw = cv2.Laplacian(img, cv2.CV_64F, ksize=3)
laplacian_raw_8bit = cv2.convertScaleAbs(laplacian_raw)
# Kenarları netleştirmek için Threshold (Eşikleme) uygulanır
_, laplacian_raw_thresh = cv2.threshold(laplacian_raw_8bit, 50, 255, cv2.THRESH_BINARY)

# 3.2. Gauss Uygulanmış Görüntü Üzerinde Laplacian (İdeal Kullanım)
laplacian_blur = cv2.Laplacian(blurred, cv2.CV_64F, ksize=3)
laplacian_blur_8bit = cv2.convertScaleAbs(laplacian_blur)
_, laplacian_blur_thresh = cv2.threshold(laplacian_blur_8bit, 50, 255, cv2.THRESH_BINARY)

# --- AŞAMA 4: 6'LI PANEL GÖRSELLEŞTİRME ---
plt.figure(figsize=(18, 10))

# ÜST SATIR: Ham Görüntü ve Ham Laplacian Sonuçları
plt.subplot(2, 3, 1); plt.imshow(img, cmap='gray'); plt.title("1. Orijinal (Gürültülü)"); plt.axis('off')
plt.subplot(2, 3, 2); plt.imshow(laplacian_raw_8bit, cmap='gray'); plt.title("2. Laplacian (Ham)"); plt.axis('off')
plt.subplot(2, 3, 3); plt.imshow(laplacian_raw_thresh, cmap='gray'); plt.title("3. Laplacian + Threshold (Ham)"); plt.axis('off')

# ALT SATIR: Gauss Blur ve Yumuşatılmış Laplacian Sonuçları
plt.subplot(2, 3, 4); plt.imshow(blurred, cmap='gray'); plt.title("4. Gauss Blur Uygulanmış"); plt.axis('off')
plt.subplot(2, 3, 5); plt.imshow(laplacian_blur_8bit, cmap='gray'); plt.title("5. Gauss + Laplacian"); plt.axis('off')
plt.subplot(2, 3, 6); plt.imshow(laplacian_blur_thresh, cmap='gray'); plt.title("6. Gauss + Laplacian + Threshold"); plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# --- AŞAMA 1: GÖRÜNTÜYÜ OKU ---
img = cv2.imread('ornek_goruntu.png', cv2.IMREAD_GRAYSCALE)

if img is None:
    # Test için gürültülü sentetik görüntü
    img = np.zeros((300, 300), dtype=np.uint8)
    cv2.rectangle(img, (80, 80), (220, 220), 180, -1)
    noise = np.random.randint(0, 35, (300, 300), dtype=np.uint8)
    img = cv2.add(img, noise)

# --- AŞAMA 2: ÖN HAZIRLIK ---
# Canny kendi içinde pürüzsüzleştirme yapsa da, önceden uygulanmış 
# bir Gauss filtresi gürültü direncini ciddi oranda artırır.
blurred = cv2.GaussianBlur(img, (5, 5), 0)

# --- AŞAMA 3: CANNY OPERASYONLARI ---
# Parametreler: cv2.Canny(görüntü, alt_esik, ust_esik)
# - ust_esik üzerindeki pikseller kesin kenardır.
# - alt_esik altındaki pikseller reddedilir.
# - İkisi arasında kalanlar, kesin bir kenara bağlıysa kabul edilir.

# 3.1. Ham Görüntü Üzerinde Canny (Gürültüye Duyarlılık Testi)
canny_raw = cv2.Canny(img, 50, 150)

# 3.2. Gauss Uygulanmış Görüntü Üzerinde Standart Canny
canny_blur_standard = cv2.Canny(blurred, 50, 150)

# 3.3. Histerezis Eşik Değerlerinin Etkisi
canny_blur_low_thresh = cv2.Canny(blurred, 10, 50)   # Dar aralık: Fazla detay/gürültü yakalar
canny_blur_high_thresh = cv2.Canny(blurred, 150, 200) # Yüksek eşik: Sadece en keskin kenarlar

# --- AŞAMA 4: 6'LI PANEL GÖRSELLEŞTİRME ---
plt.figure(figsize=(18, 10))

# ÜST SATIR: Ham Görüntü ve Gürültü Etkisi
plt.subplot(2, 3, 1); plt.imshow(img, cmap='gray'); plt.title("1. Orijinal (Gürültülü)"); plt.axis('off')
plt.subplot(2, 3, 2); plt.imshow(canny_raw, cmap='gray'); plt.title("2. Canny (Ham Görüntü)"); plt.axis('off')
plt.subplot(2, 3, 3); plt.imshow(blurred, cmap='gray'); plt.title("3. Gauss Blur Uygulanmış"); plt.axis('off')

# ALT SATIR: Eşik Değerlerinin Karşılaştırması
plt.subplot(2, 3, 4); plt.imshow(canny_blur_standard, cmap='gray'); plt.title("4. Canny Standart (50, 150)"); plt.axis('off')
plt.subplot(2, 3, 5); plt.imshow(canny_blur_low_thresh, cmap='gray'); plt.title("5. Canny Düşük Eşik (10, 50)"); plt.axis('off')
plt.subplot(2, 3, 6); plt.imshow(canny_blur_high_thresh, cmap='gray'); plt.title("6. Canny Yüksek Eşik (150, 200)"); plt.axis('off')

plt.tight_layout()
plt.show()